# CE497 · Learn sand motion directly: MPM → particle GNN

**Colab:** upload → **Runtime → Change runtime type → GPU** (T4, when available) → **Run all**. CPU also works, but training a full dynamics model takes longer than the previous constitutive-learning example. All code is here; no repository, dataset, or pretrained checkpoint is downloaded.

**One example:** a 2-D dry-sand column collapses onto a rough floor. **Same workflow:** MPM theory → particle graph → generate MPM trajectories → train a GNN → compare animations → inspect the changing graph.

The GNN now learns **each particle's complete 2-D acceleration directly from position histories**, following the GNS workflow of Sanchez-Gonzalez et al. (2020) [1–3]. The MPM solver is used only to generate data and the reference movie.

This is a **small, independently implemented PyTorch GNS**, not DeepMind's released model/checkpoint or a reproduction of its reported accuracy. For a manageable classroom run, this version uses a **240-point test column**, one fixed material, 64 training trajectories, and a smaller network. Expand the six code cells to inspect the implementation.


## 1 · MPM: the numerical teacher

**Material points represent small parcels of a sand continuum—not individual grains.** They carry position, velocity, mass, and deformation history. A temporary background grid is used to solve momentum; it is cleared each substep, so the grid does not become tangled when the column collapses. The teacher uses the same simplified frictional elastoplastic sand law as the previous notebook. [4]

The continuum equation is
$$\rho\frac{D\mathbf v}{Dt}=\nabla\!\cdot\boldsymbol\sigma+\rho\mathbf g.$$
Gravity drives collapse; internal stress and floor friction resist it. In this teaching material, compression permits frictional shear resistance, excessive shear produces permanent deformation, and bulk tension is not sustained. It is **not** calibrated soil: there is no cohesion, pore water, hardening, or full 3-D constitutive model.

### One MPM substep
Let $w_{ap}$ connect material point $p$ to grid node $a$, and $\mathbf r_{ap}=\mathbf x_a-\mathbf x_p$. Quadratic B-splines give each point nine support nodes. The MLS/APIC transfers used here are:

**Particle → grid:**
$$m_a=\sum_p w_{ap}m_p,$$
$$\mathbf P_a=\sum_p w_{ap}\left[m_p(\mathbf v_p+\mathbf C_p\mathbf r_{ap})
-\frac{4\delta t V_p^0}{\Delta x^2}\boldsymbol\tau_p\mathbf r_{ap}\right].$$
$\mathbf C_p$ represents local velocity variation; $\boldsymbol\tau_p$ is Kirchhoff stress, obtained from the teacher's deformation history and elastic/plastic update. $V_p^0$ is reference volume (area per unit thickness in 2-D).

**Grid update:** $\mathbf u_a=\mathbf P_a/m_a+\delta t\mathbf g$, then apply wall contact and floor friction.

**Grid → particle:**
$$\mathbf v_p^{n+1}=\sum_a w_{ap}\mathbf u_a,\qquad
\mathbf C_p^{n+1}=\frac4{\Delta x^2}\sum_a w_{ap}\mathbf u_a\mathbf r_{ap}^{T},\qquad
\mathbf x_p^{n+1}=\mathbf x_p^n+\delta t\mathbf v_p^{n+1}.$$

```text
Update elastic/plastic stress → P2G → grid momentum + contact → G2P → move
```

**Only saved positions leave this teacher.** Stress, deformation gradients, grid velocities, and material-law parameters are never GNN inputs or targets. The teacher substep is $\delta t=0.0004$; positions are saved every 25 substeps, giving a **GNN timestep $\Delta T=0.01$** in model units.

In [1]:
#@title 1 · Define the MPM sand-column solver
import time, sys, subprocess, importlib.util, math, json, uuid, copy
START = time.perf_counter()
missing = [p for p in ('numpy', 'scipy', 'numba', 'torch')
           if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import numpy as np
import torch
from torch import nn
from numba import njit
from IPython.display import HTML, display
np.random.seed(12); torch.manual_seed(12); torch.set_num_threads(2)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Change these before Run all. The same material is used in training and testing.
N_TRAIN = 64 #@param {type:"integer"}
MAX_UPDATES = 6000 #@param {type:"integer"}
BETA = 0.60 #@param {type:"number"}
assert N_TRAIN >= 2 and MAX_UPDATES >= 100 and 0 < BETA < 1
N_VALID = 4
DX = .04; D = DX/2       # four material points per initially filled grid cell
DT = .0004; STEPS = 1800; SAVE = 25
GX, GY = 31, 25
BOX = np.array([(GX-1)*DX, (GY-1)*DX])
RHO = 1.; VOL = D*D; MASS = RHO*VOL
K = 400.; MU = 200.; WALL_MU = .5
assert STEPS % SAVE == 0

@njit
def trial_state(F, C):
    """Elastic trial state -> pressure, trial shear, principal directions.
    The closed-form 2x2 eigensystem below is equivalent to extracting
    the left singular vectors and log singular values of the trial F.
    This analytic update belongs to the MPM teacher only.
    """
    n = len(F)
    trial = np.empty_like(F)
    z, axes, strain = np.empty((n, 2)), np.empty((n, 2)), np.empty((n, 2))
    for p in range(n):
        T = (np.eye(2) + DT*C[p]) @ F[p]
        det = T[0, 0]*T[1, 1] - T[0, 1]*T[1, 0]
        if det <= 1e-10:
            raise ValueError('Inverted deformation gradient; reduce DT. No correction is applied.')
        a = T[0, 0]**2 + T[0, 1]**2
        b = T[0, 0]*T[1, 0] + T[0, 1]*T[1, 1]
        c = T[1, 0]**2 + T[1, 1]**2
        gap = math.hypot(a-c, 2*b)
        lam1 = .5*(a+c+gap); lam2 = det*det/lam1
        t = math.log(det); d = max(0., .25*math.log(lam1/lam2))
        trial[p] = T
        z[p, 0], z[p, 1] = K*max(-t, 0.), 2*MU*d
        strain[p, 0], strain[p, 1] = t, d
        axes[p, 0] = (a-c)/gap if gap > 1e-14 else 1.
        axes[p, 1] = 2*b/gap if gap > 1e-14 else 0.
    return trial, z, axes, strain

@njit
def rebuild(trial, z, axes, strain, q):
    """Use a supplied shear stress (analytic) to update F and stress.
    The elastic volume is preserved in compression; tensile stress is zero.
    This is part of the teacher, never part of the GNN rollout.
    """
    F, tau = np.empty_like(trial), np.empty_like(trial)
    for p in range(len(trial)):
        pressure = z[p, 0]
        tnew, dnew = min(strain[p, 0], 0.), q[p]/(2*MU)
        ep = math.exp((tnew-strain[p, 0])/2 + dnew-strain[p, 1])
        em = math.exp((tnew-strain[p, 0])/2 - dnew+strain[p, 1])
        a, b = .5*(ep+em), .5*(ep-em)
        c2, s2 = axes[p]   # cos(2*theta), sin(2*theta)
        stretch_correction = np.array([[a+b*c2, b*s2], [b*s2, a-b*c2]])
        F[p] = stretch_correction @ trial[p]
        tau[p] = np.array([[-pressure+q[p]*c2, q[p]*s2],
                           [q[p]*s2, -pressure-q[p]*c2]])
    return F, tau
@njit
def particle_grid_graph(x):
    """Nine quadratic B-spline edges per material point; grid nodes are fixed."""
    n = len(x)
    nodes = np.empty((n, 9), np.int64)
    weights = np.empty((n, 9))
    offsets = np.empty((n, 9, 2))
    for p in range(n):
        bx = int(math.floor(x[p, 0] / DX - .5))
        by = int(math.floor(x[p, 1] / DX - .5))
        if bx < 0 or by < 0 or bx + 2 >= GX or by + 2 >= GY:
            raise ValueError('A particle left the grid. Reduce DT or check the reference setup.')
        fx, fy = x[p, 0] / DX - bx, x[p, 1] / DX - by
        wx = (.5*(1.5-fx)**2, .75-(fx-1)**2, .5*(fx-.5)**2)
        wy = (.5*(1.5-fy)**2, .75-(fy-1)**2, .5*(fy-.5)**2)
        for i in range(3):
            for j in range(3):
                e = 3*i + j
                nodes[p, e] = (bx+i)*GY + by+j
                weights[p, e] = wx[i]*wy[j]
                offsets[p, e, 0] = (i-fx)*DX
                offsets[p, e, 1] = (j-fy)*DX
    return nodes, weights, offsets

@njit
def transfers(x, v, C, tau):
    nodes, weights, offsets = particle_grid_graph(x)
    gm = np.zeros(GX*GY)
    gv = np.zeros((GX*GY, 2))
    factor = 4 / DX**2
    # P2G: sum mass and momentum messages arriving at each grid node.
    for p in range(len(x)):
        A = MASS*C[p] - DT*VOL*factor*tau[p]
        for e in range(9):
            a, w = nodes[p, e], weights[p, e]
            rx, ry = offsets[p, e]
            gm[a] += w*MASS
            gv[a, 0] += w*(MASS*v[p, 0] + A[0, 0]*rx + A[0, 1]*ry)
            gv[a, 1] += w*(MASS*v[p, 1] + A[1, 0]*rx + A[1, 1]*ry)
    # Grid update: momentum -> velocity, gravity, and frictional floor contact.
    for a in range(GX*GY):
        if gm[a] > 0:
            i, j = a//GY, a%GY
            gv[a] /= gm[a]
            gv[a, 1] -= DT*9.81
            if j <= 2 and gv[a, 1] < 0:
                vx, impulse = gv[a, 0], -gv[a, 1]
                gv[a, 0] = math.copysign(max(0., abs(vx)-WALL_MU*impulse), vx)
                gv[a, 1] = 0.
            if i <= 2: gv[a, 0] = max(gv[a, 0], 0.)
            if i >= GX-3: gv[a, 0] = min(gv[a, 0], 0.)
            if j >= GY-3: gv[a, 1] = min(gv[a, 1], 0.)
    # G2P: interpolate updated velocities and the local affine velocity matrix.
    vn, Cn = np.zeros_like(v), np.zeros_like(C)
    for p in range(len(x)):
        for e in range(9):
            a, w = nodes[p, e], weights[p, e]
            rx, ry = offsets[p, e]
            ux, uy = gv[a]
            vn[p, 0] += w*ux
            vn[p, 1] += w*uy
            Cn[p, 0, 0] += factor*w*ux*rx
            Cn[p, 0, 1] += factor*w*ux*ry
            Cn[p, 1, 0] += factor*w*uy*rx
            Cn[p, 1, 1] += factor*w*uy*ry
    return x + DT*vn, vn, Cn


@njit
def simulate(x):
    """Reference MPM teacher; only its particle-position trajectory is retained."""
    v = np.zeros_like(x); C = np.zeros((len(x), 2, 2)); F = np.empty_like(C)
    for p in range(len(x)): F[p] = np.eye(2)
    path = np.empty((STEPS//SAVE+1, len(x), 2)); path[0] = x
    for s in range(STEPS):
        trial, z, axes, strain = trial_state(F, C)
        q = np.minimum(z[:, 1], BETA*z[:, 0])  # analytic MPM teacher ONLY
        F, tau = rebuild(trial, z, axes, strain, q)
        x, v, C = transfers(x, v, C, tau)
        if (s+1) % SAVE == 0:
            path[(s+1)//SAVE] = x
    return path

def column(seed, test=False):
    rng = np.random.default_rng(seed)
    nx, ny = 12, 20
    if not test:
        while (nx, ny) == (12, 20):  # reserve this exact size for the final test
            nx, ny = int(rng.integers(10, 15)), int(rng.integers(18, 24))
    x = np.stack(np.meshgrid(np.arange(nx)*D, np.arange(ny)*D), -1).reshape(-1, 2)
    x += np.array([.6-(nx-1)*D/2, .105])
    x += rng.uniform(-.15*D, .15*D, x.shape)
    return x

print(f'Training device: {DEVICE}. MPM runs on CPU; the GNN uses this device.')
print('The first simulation includes one-time Numba compilation.')

Training device: cuda. MPM runs on CPU; the GNN uses this device.
The first simulation includes one-time Numba compilation.


## 2 · Direct particle-acceleration GNN

The GNN has **no MPM grid**. One graph node is one material point; directed edges connect points within radius $R$, and the graph is rebuilt from the current positions. Self-edges are also included. [2]

### Inputs, targets, and motion update
Six recent position frames give five finite-difference velocities:
$$\mathbf v_t=\frac{\mathbf x_t-\mathbf x_{t-1}}{\Delta T}.$$
The supervised target is the effective acceleration over the **saved-frame timestep**:
$$\boxed{\mathbf a_t^{\mathrm{target}}=
\frac{\mathbf x_{t+1}^{\mathrm{MPM}}-2\mathbf x_t^{\mathrm{MPM}}+\mathbf x_{t-1}^{\mathrm{MPM}}}{\Delta T^2}.}$$
This is derived from particle positions, **not** from an MPM stress formula or an instantaneous substep force.

**Node features:** five velocities plus clipped distances to the four walls. **Edge features:** relative displacement divided by $R$, and its length. All points have the same material, so a material-type embedding is unnecessary.

```text
6 position frames → velocity histories + current radius graph
       ↓
Node/edge encoders → 4 rounds of learned message passing → node decoder
       ↓
2-D acceleration for EVERY particle
       ↓
v_next = v_current + ΔT × acceleration
x_next = x_current + ΔT × v_next
```

Within a processor block, learned edge messages are summed at each receiving node, then its latent state is updated. Residual connections and layer normalization are used. The decoder outputs normalized $(a_x,a_y)$; the same trained network is reused at every simulation timestep. [1, 2]

**After initialization, the GNN must learn the effects of gravity, internal forces, and contact from the trajectories.** Its only explicit motion rule is the kinematic update above—no analytic sand force, gravity addition, wall projection, or MPM correction is used.

In [2]:
#@title 2 · Define the direct particle-acceleration GNN
import numpy as np, torch, math
from torch import nn
from scipy.spatial import cKDTree
HISTORY=6
RADIUS=.065
NOISE_VEL=.03  # noise standard deviation of the final input velocity
FRAME_DT=DT*SAVE  # saved-frame interval; not the smaller MPM substep
LOW=np.array([.08,.08],np.float32)
HIGH=np.array([1.12,.88],np.float32)

def radius_graph(x):
    """Directed nearby-particle edges, rebuilt from CURRENT positions only."""
    pairs=cKDTree(x).query_pairs(RADIUS,output_type='ndarray')
    i,j=pairs.T
    self_ids=np.arange(len(x),dtype=np.int64)
    send=np.concatenate((i,j,self_ids))
    recv=np.concatenate((j,i,self_ids))
    return send,recv

def mlp(inp,out,width=48,norm=True):
    layers=[nn.Linear(inp,width),nn.ReLU(),nn.Linear(width,width),nn.ReLU(),nn.Linear(width,out)]
    if norm:layers.append(nn.LayerNorm(out))
    return nn.Sequential(*layers)

class Processor(nn.Module):
    def __init__(self,width):
        super().__init__()
        self.edge=mlp(3*width,width,width)
        self.node=mlp(2*width,width,width)
    def forward(self,h,e,send,recv):
        de=self.edge(torch.cat((h[send],h[recv],e),dim=-1))
        agg=torch.zeros_like(h).index_add_(0,recv,de)
        dh=self.node(torch.cat((h,agg),dim=-1))
        return h+dh,e+de

class GNS(nn.Module):
    """Encode -> residual message passing -> each particle's (ax, ay)."""
    def __init__(self,stats,width=48,rounds=4):
        super().__init__()
        for k,v in stats.items():self.register_buffer(k,torch.tensor(v,dtype=torch.float32))
        self.node_encoder=mlp(2*(HISTORY-1)+4,width,width)
        self.edge_encoder=mlp(3,width,width)
        self.processor=nn.ModuleList([Processor(width) for _ in range(rounds)])
        self.decoder=mlp(width,2,width,False)
    def forward(self,seq,send,recv):
        x=seq[:,-1]
        velocity=((seq[:,1:]-seq[:,:-1])/FRAME_DT-self.vmean)/self.vstd
        boundary=torch.cat((x-self.low,self.high-x),-1).div(RADIUS).clamp(-1,1)
        h=self.node_encoder(torch.cat((velocity.flatten(1),boundary),-1))
        dr=(x[send]-x[recv])/RADIUS
        e=self.edge_encoder(torch.cat((dr,dr.norm(dim=-1,keepdim=True)),-1))
        for block in self.processor:h,e=block(h,e,send,recv)
        return self.decoder(h)


def make_example(path,t,rng,noise=True):
    """One position-history window and its acceleration target; no stress labels."""
    seq=path[t-HISTORY+1:t+1].transpose(1,0,2).copy()
    nxt=path[t+1].copy()
    if noise:
        dv_noise=rng.normal(0,NOISE_VEL*FRAME_DT/math.sqrt(HISTORY-1),(len(seq),HISTORY-1,2)).astype(np.float32).cumsum(1)
        pos_noise=np.concatenate((np.zeros((len(seq),1,2),np.float32),dv_noise.cumsum(1)),1)
        seq+=pos_noise
        # GNS target adjustment: correct velocity noise, not absolute position noise.
        nxt+=pos_noise[:,-1]
    # Finite-difference acceleration in physical/model-time units.
    acc=(nxt-2*seq[:,-1]+seq[:,-2])/FRAME_DT**2
    send,recv=radius_graph(seq[:,-1])
    return seq,send,recv,acc

@torch.inference_mode()
def rollout(model,seed_history,n_frames):
    """Only the initial six frames are provided. No reference/MPM state is read."""
    if len(seed_history)!=HISTORY:
        raise ValueError(f'Provide exactly {HISTORY} starting frames.')
    device=next(model.parameters()).device
    past=np.array(seed_history,dtype=np.float32,copy=True)
    out=list(past.copy())
    for step in range(n_frames-len(past)):
        send,recv=radius_graph(past[-1])
        seq=torch.tensor(past[-HISTORY:].transpose(1,0,2),device=device)
        a_norm=model(seq,torch.tensor(send,device=device),torch.tensor(recv,device=device))
        a=(a_norm*model.astd+model.amean).cpu().numpy()
        # Equivalent to v_next=v_current+dt*a; x_next=x_current+dt*v_next.
        xnew=2*past[-1]-past[-2]+FRAME_DT**2*a
        if not np.isfinite(xnew).all():raise RuntimeError('Nonfinite GNN rollout; no MPM fallback applied.')
        out.append(xnew)
        past=np.concatenate((past[-HISTORY+1:],xnew[None]),0)
    return np.stack(out)

print("Graph nodes: material points only; radius:", RADIUS)
print("Inputs: 5 velocities + wall distances. Output: (ax, ay) per particle.")

Graph nodes: material points only; radius: 0.065
Inputs: 5 velocities + wall distances. Output: (ax, ay) per particle.


## 3 · Generate MPM position trajectories

Generate columns with slightly different widths, heights, and initial particle offsets. Split by **whole trajectory**, not by randomly mixing neighboring frames from the same movie.

**Training:** 64 trajectories. **Validation:** 4 separate initializations. **Final comparison:** a new $12\times20$ column and seed 987; this exact geometry is excluded from training and validation. Normalization statistics are calculated from training trajectories only.

Each training example is **six position frames → next-step particle accelerations**. Thousands of overlapping windows are useful training examples, but they are not thousands of independent physical experiments.

In [3]:
#@title 3 · Generate MPM trajectories and compute training-only normalization
train_paths, valid_paths = [], []
data_start = time.perf_counter()
for k in range(N_TRAIN + N_VALID):
    seed = k if k < N_TRAIN else 10000 + k
    path = simulate(column(seed)).astype(np.float32)
    if not np.isfinite(path).all():
        raise RuntimeError('Non-finite MPM trajectory; check the teacher parameters.')
    (train_paths if k < N_TRAIN else valid_paths).append(path)
    if (k+1) % 8 == 0 or k+1 == N_TRAIN + N_VALID:
        print(f'Generated {k+1}/{N_TRAIN+N_VALID} MPM trajectories.')

# Only positions are stored. These are velocities/accelerations of SAVED frames.
velocities = np.concatenate([np.diff(p, axis=0).reshape(-1, 2)/FRAME_DT
                             for p in train_paths])
accelerations = np.concatenate([np.diff(p, n=2, axis=0).reshape(-1, 2)/FRAME_DT**2
                                for p in train_paths])
stats = dict(vmean=velocities.mean(0),
             vstd=np.sqrt(velocities.var(0) + NOISE_VEL**2),
             amean=accelerations.mean(0),
             astd=np.sqrt(accelerations.var(0) + (NOISE_VEL/FRAME_DT)**2),
             low=LOW, high=HIGH)
del velocities, accelerations
n_windows = sum(len(p)-HISTORY for p in train_paths)
print(f'{N_TRAIN} training / {N_VALID} validation trajectories; {n_windows:,} training windows.')
print(f'GNN timestep: {FRAME_DT:g}; saved duration: {STEPS*DT:g}. '
      f'Data generation: {time.perf_counter()-data_start:.1f} s.')

Generated 8/68 MPM trajectories.
Generated 16/68 MPM trajectories.
Generated 24/68 MPM trajectories.
Generated 32/68 MPM trajectories.
Generated 40/68 MPM trajectories.
Generated 48/68 MPM trajectories.
Generated 56/68 MPM trajectories.
Generated 64/68 MPM trajectories.
Generated 68/68 MPM trajectories.
64 training / 4 validation trajectories; 4,288 training windows.
GNN timestep: 0.01; saved duration: 0.72. Data generation: 60.5 s.


## 4 · Train from scratch

The network predicts **accelerations**, not material parameters. Adam minimizes normalized acceleration mean-square error:
$$\mathcal L=\frac1{2N}\sum_{p=1}^N
\left\|\widehat{\mathbf a}_{p,\mathrm{norm}}-\mathbf a_{p,\mathrm{norm}}^{\mathrm{target}}\right\|^2.$$

Small **random-walk velocity noise** is integrated into the input position histories. As in the reference GNS implementation, the next-position target is shifted by the final position noise. This trains recovery from velocity errors without forcing the network to undo arbitrary position offsets. The graph is rebuilt from the noisy positions. [3]

Weights start randomly. Only validation trajectories are used to select the checkpoint; the final test movie is untouched. Printed losses are training diagnostics, not a separate error-analysis exercise. **No time limit silently shortens training.**

In [4]:
#@title 4 · Train on particle accelerations, not shear/material coefficients
# Reset seeds here so re-running this cell starts the same experiment.
torch.manual_seed(12)
rng = np.random.default_rng(24)
model = GNS(stats).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=.001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, MAX_UPDATES, eta_min=.00005)
print(f'{sum(p.numel() for p in model.parameters()):,} trainable parameters; '
      f'{MAX_UPDATES:,} updates on {DEVICE}.')

# Fixed noisy windows from VALIDATION trajectories only.
vrng = np.random.default_rng(999)
validation_examples = []
for p in valid_paths:
    for t in (6, 12, 20, 30, 45, 60):
        validation_examples.append([torch.as_tensor(z, device=DEVICE)
                                    for z in make_example(p, t, vrng)])
best_score, best_weights, best_step = float('inf'), None, 0
train_start = time.perf_counter()
for step in range(MAX_UPDATES):
    path = train_paths[int(rng.integers(len(train_paths)))]
    t = int(rng.integers(HISTORY-1, len(path)-1))
    seq, send, recv, target = [torch.as_tensor(z, device=DEVICE)
                             for z in make_example(path, t, rng)]
    prediction = model(seq, send, recv)
    target_normalized = (target-model.amean)/model.astd
    loss = (prediction-target_normalized).square().mean()
    if not torch.isfinite(loss):
        raise RuntimeError('Non-finite training loss; no fallback model is substituted.')
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    if (step+1) % 1000 == 0 or step+1 == MAX_UPDATES:
        model.eval()
        with torch.inference_mode():
            val_loss = np.mean([
                (model(s, i, j)-(a-model.amean)/model.astd).square().mean().item()
                for s, i, j, a in validation_examples])
            # Select weights by autonomous VALIDATION rollouts, not the final test.
            # This is checkpoint selection only; training still uses one-step acceleration MSE.
            score = 0.0
            for p in valid_paths:
                vpred = rollout(model, p[:HISTORY], len(p))
                score += float(np.mean((vpred[HISTORY:]-p[HISTORY:])**2))
            score /= len(valid_paths)
        if np.isfinite(score) and score < best_score:
            best_score, best_step = score, step+1
            best_weights = copy.deepcopy(model.state_dict())
        print(f'Update {step+1:>6,}: acceleration loss {loss.item():.4f}; '
              f'validation acceleration loss {val_loss:.4f}; '
              f'elapsed {(time.perf_counter()-train_start)/60:.1f} min.')
        model.train()

if best_weights is None:
    raise RuntimeError('No finite validation checkpoint was produced.')
model.load_state_dict(best_weights)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
torch.save({'state_dict': model.cpu().state_dict(),
            'stats': {k: np.asarray(v).tolist() for k, v in stats.items()},
            'history': HISTORY, 'frame_dt': FRAME_DT, 'radius': RADIUS,
            'width': 48, 'message_rounds': 4}, 'mpm_particle_gns.pt')
model.to(DEVICE)
print(f'Selected update {best_step:,} using validation trajectories. '
      f'Saved weights: mpm_particle_gns.pt. Training: '
      f'{(time.perf_counter()-train_start)/60:.1f} min.')

100,178 trainable parameters; 6,000 updates on cuda.
Update  1,000: acceleration loss 0.1367; validation acceleration loss 0.2363; elapsed 0.3 min.
Update  2,000: acceleration loss 0.1365; validation acceleration loss 0.1301; elapsed 0.5 min.
Update  3,000: acceleration loss 0.1511; validation acceleration loss 0.0931; elapsed 0.8 min.
Update  4,000: acceleration loss 0.0178; validation acceleration loss 0.0869; elapsed 1.0 min.
Update  5,000: acceleration loss 0.0044; validation acceleration loss 0.0718; elapsed 1.3 min.
Update  6,000: acceleration loss 0.0666; validation acceleration loss 0.0625; elapsed 1.6 min.
Selected update 6,000 using validation trajectories. Saved weights: mpm_particle_gns.pt. Training: 1.6 min.


## 5 · Held-out comparison: MPM vs autonomous GNN

**Left:** the reference MPM trajectory. **Right:** direct GNN acceleration predictions followed by time integration.

The first **six MPM position frames** ($t=0$ through $0.05$) initialize the GNN history, as in a history-based GNS rollout. They are labeled **given history**, not predictions. From frame 6 onward, the GNN uses **its own predicted positions only**. There is no reset, blending, MPM fallback, or hidden constitutive-law call.

The two panels have identical time, scale, particle count, and particle IDs. Their differences show accumulated learned-model error; the right panel is not a replay of the left.

In [5]:
#@title 5 · Compare MPM with a genuinely independent particle-GNN rollout
x0 = column(987, test=True)  # this geometry and seed were not used above
truth = simulate(x0.copy()).astype(np.float32)
predicted = rollout(model, truth[:HISTORY].copy(), len(truth))
assert truth.shape == predicted.shape
assert np.array_equal(truth[:HISTORY], predicted[:HISTORY])
print(f'{len(x0)} material points; {HISTORY} given frames; '
      f'{len(truth)-HISTORY} autonomous GNN prediction steps.')
print('The GNN rollout calls no MPM functions and applies no analytic forces or wall clipping.')

# A common view includes ALL points, so an escaping prediction is never silently hidden.
all_points = np.concatenate((truth.reshape(-1, 2), predicted.reshape(-1, 2)))
VIEW = [float(min(0., all_points[:, 0].min()-.02)),
        float(max(BOX[0], all_points[:, 0].max()+.02)),
        float(min(0., all_points[:, 1].min()-.02)),
        float(max(.72, all_points[:, 1].max()+.02))]
uid = 'sand_direct_' + uuid.uuid4().hex[:10]
payload = json.dumps({'truth': truth.round(6).tolist(), 'gnn': predicted.round(6).tolist(),
                      'view': VIEW, 'dt': FRAME_DT, 'floor': 2*DX, 'history': HISTORY},
                     separators=(',', ':'))
html = r'''
<div id="__ID__" style="font-family:system-ui;max-width:1000px;padding:14px;background:#111;color:#fff;border-radius:10px">
 <div style="display:flex;gap:16px;flex-wrap:wrap">
  <div style="flex:1;min-width:280px"><h3>MPM reference</h3>
   <canvas data-name="truth" width="640" height="400" style="width:100%;background:#000;border:1px solid #666"></canvas></div>
  <div style="flex:1;min-width:280px"><h3>GNN · direct acceleration prediction</h3>
   <canvas data-name="gnn" width="640" height="400" style="width:100%;background:#000;border:1px solid #666"></canvas></div>
 </div>
 <div style="display:flex;align-items:center;gap:12px;margin-top:12px;flex-wrap:wrap">
  <button type="button">Pause</button><input aria-label="Simulation frame" type="range" min="0" value="0" style="flex:1">
  <span data-time style="min-width:105px"></span>
 </div>
 <div data-status style="font-size:14px;margin-top:10px;font-weight:600"></div>
 <p style="font-size:13px;margin-bottom:0">Same particle IDs and view. Only the first six frames are given to the GNN. Afterwards, it advances its own position history. Playback is slowed; t is model time.</p>
</div>
<script>
(()=>{
 const root=document.getElementById('__ID__'), data=__DATA__;
 const slider=root.querySelector('input'), button=root.querySelector('button'), text=root.querySelector('[data-time]'), status=root.querySelector('[data-status]');
 let frame=0,playing=true,last=0; slider.max=data.truth.length-1;
 function draw(){
  for(const c of root.querySelectorAll('canvas')){
   const ctx=c.getContext('2d'),pts=data[c.dataset.name][frame],v=data.view;
   const scale=Math.min((c.width-24)/(v[1]-v[0]),(c.height-24)/(v[3]-v[2]));
   const ox=(c.width-scale*(v[1]-v[0]))/2,oy=(c.height-scale*(v[3]-v[2]))/2;
   const X=x=>ox+(x-v[0])*scale,Y=y=>c.height-oy-(y-v[2])*scale;
   ctx.fillStyle='#000';ctx.fillRect(0,0,c.width,c.height);
   const floorY=Y(data.floor);ctx.fillStyle='#222';ctx.fillRect(X(v[0]),floorY,scale*(v[1]-v[0]),Math.max(0,Y(v[2])-floorY));
   ctx.beginPath();ctx.moveTo(X(v[0]),floorY);ctx.lineTo(X(v[1]),floorY);ctx.strokeStyle='#777';ctx.stroke();
   ctx.fillStyle='#fff';
   for(const p of pts){ctx.beginPath();ctx.arc(X(p[0]),Y(p[1]),2.5,0,2*Math.PI);ctx.fill();}
  }
  slider.value=frame;text.textContent='t = '+(frame*data.dt).toFixed(3);
  status.textContent=frame<data.history?'GNN: given initialization history (frames 0–5)':'GNN: autonomous prediction — no reference correction';
 }
 button.onclick=()=>{playing=!playing;button.textContent=playing?'Pause':'Play';};
 slider.oninput=()=>{frame=+slider.value;playing=false;button.textContent='Play';draw();};
 function tick(now){if(!root.isConnected)return;if(playing&&now-last>50){frame=(frame+1)%data.truth.length;draw();last=now;}requestAnimationFrame(tick);}
 draw();requestAnimationFrame(tick);
})();
</script>
'''.replace('__ID__', uid).replace('__DATA__', payload)
display(HTML(html))

240 material points; 6 given frames; 67 autonomous GNN prediction steps.
The GNN rollout calls no MPM functions and applies no analytic forces or wall clipping.


## 6 · Watch the learned simulator's graph change

These are the **actual particle–particle radius connections** from the GNN rollout—not the teacher's particle–grid stencil. Highlighted edges show one point's current neighborhood. For readability, each bidirectional pair is drawn once and self-edges are omitted from the drawing only. The network uses all edges and performs four message-passing rounds per physical prediction step.

In [6]:
#@title 6 · Animate the actual particle-to-particle GNN graph
# Use the same radius_graph() function as inference, before rounding displayed positions.
edge_frames = []
for x in predicted:
    send, recv = radius_graph(x)
    keep = send < recv  # draw each bidirectional pair once; omit self-lines only in display
    edge_frames.append(np.stack((send[keep], recv[keep]), -1).tolist())
highlight = int(np.argmin(np.sum((x0-np.array([.6, .30]))**2, axis=1)))
uid2 = 'sand_particle_graph_' + uuid.uuid4().hex[:10]
graph_payload = json.dumps({'gnn': predicted.round(6).tolist(), 'edges': edge_frames,
                            'view': VIEW, 'dt': FRAME_DT, 'floor': 2*DX,
                            'hi': highlight, 'history': HISTORY}, separators=(',', ':'))
html2 = r'''
<div id="__ID__" style="font-family:system-ui;max-width:850px;padding:14px;background:#111;color:#fff;border-radius:10px">
 <h3>GNN · changing particle–particle graph</h3>
 <canvas width="840" height="520" style="width:100%;background:#000;border:1px solid #666"></canvas>
 <div style="display:flex;align-items:center;gap:12px;margin-top:10px;flex-wrap:wrap">
  <button type="button">Pause</button><input data-frame aria-label="Graph frame" type="range" min="0" value="0" style="flex:1"><span data-time style="min-width:105px"></span>
 </div>
 <div style="font-size:13px;margin-top:10px">
  <span style="color:white">●</span> material point &nbsp;
  <span style="color:#36d7ff">─</span> nearby particle pair &nbsp;
  <span style="color:#ffd84d">● ─</span> selected point + its neighbors
  <label style="display:block;margin-top:8px"><input data-edges type="checkbox" checked> Show all radius-graph edges</label>
  <div data-status style="margin-top:8px"></div>
 </div>
</div>
<script>
(()=>{
 const root=document.getElementById('__ID__'),data=__DATA__,c=root.querySelector('canvas'),ctx=c.getContext('2d');
 const slider=root.querySelector('[data-frame]'),toggle=root.querySelector('[data-edges]'),button=root.querySelector('button'),text=root.querySelector('[data-time]'),status=root.querySelector('[data-status]');
 const v=data.view,scale=Math.min((c.width-24)/(v[1]-v[0]),(c.height-24)/(v[3]-v[2]));
 const ox=(c.width-scale*(v[1]-v[0]))/2,oy=(c.height-scale*(v[3]-v[2]))/2;
 const X=p=>ox+(p[0]-v[0])*scale,Y=p=>c.height-oy-(p[1]-v[2])*scale;
 let frame=0,playing=true,last=0;slider.max=data.gnn.length-1;
 function draw(){
  const pts=data.gnn[frame],edges=data.edges[frame],neighbors=new Set(),hi=data.hi;
  ctx.fillStyle='#000';ctx.fillRect(0,0,c.width,c.height);
  const floorY=Y([0,data.floor]);ctx.fillStyle='#222';ctx.fillRect(X([v[0],0]),floorY,scale*(v[1]-v[0]),Math.max(0,Y([0,v[2]])-floorY));
  ctx.beginPath();
  for(const [i,j] of edges){
   if(i===hi)neighbors.add(j);if(j===hi)neighbors.add(i);
   if(toggle.checked){ctx.moveTo(X(pts[i]),Y(pts[i]));ctx.lineTo(X(pts[j]),Y(pts[j]));}
  }
  ctx.strokeStyle='rgba(54,215,255,.16)';ctx.lineWidth=.7;ctx.stroke();
  ctx.beginPath();for(const j of neighbors){ctx.moveTo(X(pts[hi]),Y(pts[hi]));ctx.lineTo(X(pts[j]),Y(pts[j]));}
  ctx.strokeStyle='#ffd84d';ctx.lineWidth=1.8;ctx.stroke();
  ctx.fillStyle='#fff';for(const p of pts){ctx.beginPath();ctx.arc(X(p),Y(p),2.6,0,2*Math.PI);ctx.fill();}
  ctx.strokeStyle='#ffd84d';ctx.lineWidth=1.5;
  for(const j of neighbors){ctx.beginPath();ctx.arc(X(pts[j]),Y(pts[j]),4,0,2*Math.PI);ctx.stroke();}
  ctx.fillStyle='#ffd84d';ctx.beginPath();ctx.arc(X(pts[hi]),Y(pts[hi]),5,0,2*Math.PI);ctx.fill();
  slider.value=frame;text.textContent='t = '+(frame*data.dt).toFixed(3);
  status.textContent=(frame<data.history?'Given history':'Autonomous GNN')+' · '+edges.length+' nearby pairs · '+neighbors.size+' neighbors of selected point';
 }
 button.onclick=()=>{playing=!playing;button.textContent=playing?'Pause':'Play';};
 slider.oninput=()=>{frame=+slider.value;playing=false;button.textContent='Play';draw();};toggle.onchange=draw;
 function tick(now){if(!root.isConnected)return;if(playing&&now-last>50){frame=(frame+1)%data.gnn.length;draw();last=now;}requestAnimationFrame(tick);}
 draw();requestAnimationFrame(tick);
})();
</script>
'''.replace('__ID__', uid2).replace('__DATA__', graph_payload)
display(HTML(html2))

### References

**[1]** Sanchez-Gonzalez et al. (2020), [*Learning to Simulate Complex Physics with Graph Networks*](https://proceedings.mlr.press/v119/sanchez-gonzalez20a.html), ICML. Basis for the acceleration-learning, encode–process–decode workflow. This notebook is smaller and uses its own MPM dataset, not the paper's benchmark dataset or weights.

**[2]** DeepMind's reference implementation: [learned simulator](https://github.com/google-deepmind/deepmind-research/blob/master/learning_to_simulate/learned_simulator.py) and [graph network](https://github.com/google-deepmind/deepmind-research/blob/master/learning_to_simulate/graph_network.py). This notebook independently implements the corresponding operations in PyTorch; it does not import those files.

**[3]** DeepMind's [random-walk noise implementation](https://github.com/google-deepmind/deepmind-research/blob/master/learning_to_simulate/noise_utils.py) and acceleration-target adjustment in its learned simulator.

**[4]** The MPM teacher is retained from the supplied `CE497_MPM_GNN_Sand_Column.ipynb`, with fewer material points and a different saved-frame interval. Its MLS/APIC transfers follow the framework of [Hu et al. (2018)](https://yuanming.taichi.graphics/publication/2018-mlsmpm/); its simplified frictional law is related to Klár et al. (2016), *Drucker–Prager Elastoplasticity for Sand Animation*, DOI: 10.1145/2897824.2925906, but is not that complete constitutive model.